<a href="https://colab.research.google.com/github/ranjani-cse/20m-llm-reversible/blob/main/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0))
print("free GB:", torch.cuda.mem_get_info()[0]/1e9)

torch: 2.11.0+cu128 | cuda: True
device: Tesla T4
free GB: 15.5271168


In [3]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB / {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")

free: 15.53 GB / 15.64 GB


In [4]:
import os, math, time, gc, json, pickle
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset
from dataclasses import dataclass

In [6]:
@dataclass
class Config:
    vocab_size: int = 50257
    n_layer: int = 6
    n_head: int = 8
    n_embd: int = 288
    block_size: int = 512
    batch_size: int = 16
    max_steps: int = 6104
    learning_rate: float = 3e-4
    warmup_steps: int = 100
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    log_every: int = 100
    device: str = "cuda"
    seed: int = 1337

cfg = Config()
print(f"batch={cfg.batch_size}  steps={cfg.max_steps}")
print(f"tokens/step={cfg.batch_size*cfg.block_size}  total={cfg.batch_size*cfg.block_size*cfg.max_steps:,}")

batch=16  steps=6104
tokens/step=8192  total=50,003,968


In [5]:
import os
print("local:", os.path.exists("/content/tokens_50m.npy"))
print("drive:", os.path.exists("/content/drive/MyDrive/tokens_50m.npy"))

local: True
drive: True


In [ ]:
Baseline run 1 (batch 8, 12.5M tokens):
  final loss:  6.4774
  tokens/sec:  17,161
  peak memory: 6.01 GB

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.fc2 = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.ff = FeedForward(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init)
        for pn, p in self.named_parameters():
            if pn.endswith("proj.weight") or pn.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

print("GPT defined")

GPT defined


In [9]:
import gc, torch

B, T = cfg.batch_size, cfg.block_size

if "model" in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

model = GPT(cfg).to(cfg.device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    betas=(0.9, 0.95),
    weight_decay=cfg.weight_decay,
)

print(f"params: {model.num_params():,}")
print(f"B={B}, T={T}, tokens/step={B*T}")
print(f"free GB: {torch.cuda.mem_get_info()[0]/1e9:.2f}")

params: 20,609,568
B=16, T=512, tokens/step=8192
free GB: 15.44


In [11]:
import numpy as np
tokens = np.load("/content/drive/MyDrive/tokens_50m.npy").astype(np.int64)
print(f"tokens: {tokens.shape}, {tokens.dtype}")

tokens: (50000000,), int64


In [12]:
for name in ["cfg", "tokens", "GPT", "model", "optimizer", "Config"]:
    print(f"{name}: {'defined' if name in dir() else 'MISSING'}")

cfg: defined
tokens: defined
GPT: defined
model: defined
optimizer: defined
Config: defined


In [13]:
import time, numpy as np, torch

tokens_i64 = tokens
B, T = cfg.batch_size, cfg.block_size

def get_batch():
    s = np.random.randint(0, len(tokens_i64) - T - 1)
    chunk = tokens_i64[s : s + T + 1]
    x = torch.from_numpy(chunk[:-1]).long().unsqueeze(0).repeat(B, 1)
    y = torch.from_numpy(chunk[1:]).long().unsqueeze(0).repeat(B, 1)
    return x.to(cfg.device), y.to(cfg.device)

def get_lr(step):
    if step < cfg.warmup_steps:
        return cfg.learning_rate * (step + 1) / cfg.warmup_steps
    progress = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    return cfg.learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))

def train(model, max_steps, tag="run"):
    model.train()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.time()
    tokens_seen = 0
    losses = []
    step = 0
    while step < max_steps:
        x, y = get_batch()
        lr = get_lr(step)
        for g in optimizer.param_groups:
            g["lr"] = lr
        logits, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        losses.append(loss.item())
        tokens_seen += x.numel()
        step += 1
        if step % cfg.log_every == 0 or step == 1:
            torch.cuda.synchronize()
            elapsed = time.time() - t0
            tps = tokens_seen / elapsed
            mem = torch.cuda.max_memory_allocated() / 1e9
            print(f"[{tag}] step {step:5d}/{max_steps}  loss {loss.item():.4f}  "
                  f"lr {lr:.2e}  {tps:.0f} tok/s  peak mem {mem:.2f} GB")
    torch.cuda.synchronize()
    total_time = time.time() - t0
    final_tps = tokens_seen / total_time
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    final_loss = sum(losses[-20:]) / len(losses[-20:])
    print(f"\n=== {tag} done ===")
    print(f"final loss (avg last 20): {final_loss:.4f}")
    print(f"tokens/sec: {final_tps:.0f}")
    print(f"peak memory: {peak_mem:.2f} GB")
    print(f"total tokens seen: {tokens_seen:,}")
    return {"tag": tag, "final_loss": final_loss, "tokens_per_sec": final_tps,
            "peak_mem_gb": peak_mem, "tokens_seen": tokens_seen, "losses": losses}

print("train() defined")

train() defined


In [14]:
result_smoke16 = train(model, max_steps=5, tag="smoke_b16")

[smoke_b16] step     1/5  loss 10.8894  lr 3.00e-06  10264 tok/s  peak mem 7.61 GB

=== smoke_b16 done ===
final loss (avg last 20): 10.8371
tokens/sec: 16599
peak memory: 7.79 GB
total tokens seen: 40,960


In [15]:
import gc, torch

if "model" in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

model = GPT(cfg).to(cfg.device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    betas=(0.9, 0.95),
    weight_decay=cfg.weight_decay,
)
print(f"fresh model: {model.num_params():,}")
print(f"free GB: {torch.cuda.mem_get_info()[0]/1e9:.2f}")

fresh model: 20,609,568
free GB: 11.86


In [16]:
result_baseline = train(model, max_steps=6104, tag="baseline_50M")

[baseline_50M] step     1/6104  loss 10.8452  lr 3.00e-06  17572 tok/s  peak mem 7.62 GB
[baseline_50M] step   100/6104  loss 7.5955  lr 3.00e-04  19191 tok/s  peak mem 7.78 GB
[baseline_50M] step   200/6104  loss 7.1776  lr 3.00e-04  18831 tok/s  peak mem 7.78 GB
[baseline_50M] step   300/6104  loss 6.9713  lr 2.99e-04  18474 tok/s  peak mem 7.78 GB
[baseline_50M] step   400/6104  loss 6.5965  lr 2.98e-04  18172 tok/s  peak mem 7.78 GB
[baseline_50M] step   500/6104  loss 7.0342  lr 2.97e-04  17986 tok/s  peak mem 7.78 GB
[baseline_50M] step   600/6104  loss 7.1324  lr 2.95e-04  17866 tok/s  peak mem 7.78 GB
[baseline_50M] step   700/6104  loss 7.1788  lr 2.93e-04  17779 tok/s  peak mem 7.78 GB
[baseline_50M] step   800/6104  loss 6.7569  lr 2.90e-04  17712 tok/s  peak mem 7.78 GB
[baseline_50M] step   900/6104  loss 7.3950  lr 2.87e-04  17658 tok/s  peak mem 7.78 GB
[baseline_50M] step  1000/6104  loss 6.7054  lr 2.84e-04  17617 tok/s  peak mem 7.78 GB
[baseline_50M] step  1100/6104 

In [17]:
import pickle
with open("/content/drive/MyDrive/result_baseline_50M.pkl", "wb") as f:
    pickle.dump(result_baseline, f)
print("saved result_baseline_50M.pkl")

saved result_baseline_50M.pkl


In [ ]:
BASELINE — Run 1 of 3
  batch: 16
  steps: 6104
  tokens: 50,003,968
  final loss: 6.3058
  tokens/sec: 17,283
  peak memory: 7.78 GB
  time: ~2893s (~48 min)

In [18]:
def get_batch_diverse():
    xs, ys = [], []
    for _ in range(B):
        s = np.random.randint(0, len(tokens_i64) - T - 1)
        chunk = tokens_i64[s : s + T + 1]
        xs.append(torch.from_numpy(chunk[:-1]).long())
        ys.append(torch.from_numpy(chunk[1:]).long())
    return torch.stack(xs).to(cfg.device), torch.stack(ys).to(cfg.device)

In [19]:
import time
model.train()
torch.cuda.synchronize()
t0 = time.time()
for _ in range(20):
    x, y = get_batch_diverse()
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
print(f"{20*B*T/(time.time()-t0):.0f} tok/s with diverse batches")

18148 tok/s with diverse batches


In [24]:
import gc, torch

if "model_rev" in dir():
    del model_rev
gc.collect()
torch.cuda.empty_cache()

model_rev = GPTReversible(cfg).to(cfg.device)
optimizer = torch.optim.AdamW(model_rev.parameters(), lr=cfg.learning_rate,
                              betas=(0.9, 0.95), weight_decay=cfg.weight_decay)
print(f"rev params: {model_rev.num_params():,} ({model_rev.num_params()/1e6:.2f}M)")
print(f"free GB: {torch.cuda.mem_get_info()[0]/1e9:.2f}")

rev params: 17,620,128 (17.62M)
free GB: 11.88


In [25]:
result_smoke_rev = train(model_rev, max_steps=5, tag="smoke_rev")

[smoke_rev] step     1/5  loss 10.8813  lr 3.00e-06  17266 tok/s  peak mem 9.37 GB

=== smoke_rev done ===
final loss (avg last 20): 10.8562
tokens/sec: 19620
peak memory: 9.51 GB
total tokens seen: 40,960


In [26]:
import gc, torch

cfg.batch_size = 32
cfg.max_steps = 3052    # 3052 × 32 × 512 = 50,003,968 tokens

if "model_rev2" in dir():
    del model_rev2
gc.collect()
torch.cuda.empty_cache()

model_rev2 = GPTReversible(cfg).to(cfg.device)
optimizer = torch.optim.AdamW(model_rev2.parameters(), lr=cfg.learning_rate,
                              betas=(0.9, 0.95), weight_decay=cfg.weight_decay)

# refresh B, T in get_batch scope
B, T = cfg.batch_size, cfg.block_size

print(f"params: {model_rev2.num_params():,}")
print(f"batch: {B}, tokens/step: {B*T}, steps: {cfg.max_steps}, total: {B*T*cfg.max_steps:,}")
print(f"free GB: {torch.cuda.mem_get_info()[0]/1e9:.2f}")

params: 17,620,128
batch: 32, tokens/step: 16384, steps: 3052, total: 50,003,968
free GB: 11.84


In [28]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

free: 5.05 GB


In [29]:
import gc, torch

gc.collect()
torch.cuda.empty_cache()
print(f"free before: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

cfg.batch_size = 4
cfg.max_steps = 8192    # 8192 × 4 × 512 = 16.78M tokens

if "model_rev2" in dir():
    del model_rev2
gc.collect()
torch.cuda.empty_cache()

model_rev2 = GPTReversible(cfg).to(cfg.device)
optimizer = torch.optim.AdamW(model_rev2.parameters(), lr=cfg.learning_rate,
                              betas=(0.9, 0.95), weight_decay=cfg.weight_decay)
B, T = cfg.batch_size, cfg.block_size

print(f"batch={B}, steps={cfg.max_steps}, tokens={B*T*cfg.max_steps:,}")
print(f"free after build: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

free before: 5.05 GB
batch=4, steps=8192, tokens=16,777,216
free after build: 4.98 GB


In [30]:
result_smoke_b4 = train(model_rev2, max_steps=5, tag="smoke_b4")

[smoke_b4] step     1/5  loss 10.8398  lr 3.00e-06  13589 tok/s  peak mem 12.34 GB

=== smoke_b4 done ===
final loss (avg last 20): 10.8568
tokens/sec: 16900
peak memory: 12.48 GB
total tokens seen: 10,240


In [35]:
result_rev_max = train(model_rev2, max_steps=8192, tag="reversible_batch4_16M")

[reversible_batch4_16M] step     1/8192  loss 6.2989  lr 3.00e-06  13832 tok/s  peak mem 12.89 GB
[reversible_batch4_16M] step   100/8192  loss 6.1332  lr 3.00e-04  18698 tok/s  peak mem 12.89 GB
[reversible_batch4_16M] step   200/8192  loss 6.0089  lr 3.00e-04  18518 tok/s  peak mem 12.89 GB
[reversible_batch4_16M] step   300/8192  loss 6.4980  lr 3.00e-04  18313 tok/s  peak mem 12.89 GB
[reversible_batch4_16M] step   400/8192  loss 6.2258  lr 2.99e-04  18238 tok/s  peak mem 12.89 GB


KeyboardInterrupt: 

In [33]:
for name in ["result_rev_max", "result_reversible", "result_baseline", "model_rev2", "cfg"]:
    print(f"{name}: {'defined' if name in dir() else 'MISSING'}")

result_rev_max: MISSING
result_reversible: MISSING
result_baseline: defined
model_rev2: defined
cfg: defined


In [34]:
import torch
print("model_rev2 defined:", "model_rev2" in dir())
print("params:", model_rev2.num_params() if "model_rev2" in dir() else "N/A")
print("free GB:", torch.cuda.mem_get_info()[0]/1e9)
print("allocated GB:", torch.cuda.memory_allocated()/1e9)

model_rev2 defined: True
params: 17620128
free GB: 2.673672192
allocated GB: 11.085307392


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttentionHalf(nn.Module):
    def __init__(self, cfg, dim):
        super().__init__()
        self.dim = dim
        self.n_head = max(1, cfg.n_head // 2)
        self.head_dim = dim // self.n_head
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(self.dim, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class FeedForwardHalf(nn.Module):
    def __init__(self, cfg, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, 4 * dim)
        self.fc2 = nn.Linear(4 * dim, dim)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class ReversibleBlock(nn.Module):
    """
    RevNet additive coupling (simplest reversible variant).
      y1 = x1 + F(x2)
      y2 = x2 + G(y1)
    Backward: x2 = y2 - G(y1), x1 = y1 - F(x2)
    """
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % 2 == 0
        half = cfg.n_embd // 2
        self.half = half
        self.ln_F = nn.LayerNorm(half)
        self.ln_G = nn.LayerNorm(half)
        self.F_attn = CausalSelfAttentionHalf(cfg, half)
        self.F_ff   = FeedForwardHalf(cfg, half)
        self.G_attn = CausalSelfAttentionHalf(cfg, half)
        self.G_ff   = FeedForwardHalf(cfg, half)

    def F(self, x):
        h = self.ln_F(x)
        h = h + self.F_attn(h)
        h = h + self.F_ff(h)
        return h

    def G(self, x):
        h = self.ln_G(x)
        h = h + self.G_attn(h)
        h = h + self.G_ff(h)
        return h

    def forward(self, x1, x2):
        y1 = x1 + self.F(x2)
        y2 = x2 + self.G(y1)
        return y1, y2

class GPTReversible(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([ReversibleBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        x1, x2 = x.chunk(2, dim=-1)
        for block in self.blocks:
            x1, x2 = block(x1, x2)
        x = torch.cat([x1, x2], dim=-1)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

In [23]:
class ReversibleFunction(torch.autograd.Function):
    """
    Recompute activations in backward instead of storing them.
    Forward:  y1 = x1 + F(x2),  y2 = x2 + G(y1)
    Backward: x2 = y2 - G(y1),  x1 = y1 - F(x2)
    """
    @staticmethod
    def forward(ctx, x1, x2, F_module, G_module):
        with torch.no_grad():
            y1 = x1 + F_module(x2)
            y2 = x2 + G_module(y1)
        # do NOT save y1, y2 — they're reconstructible
        ctx.save_for_backward(y1, y2)
        ctx.F = F_module
        ctx.G = G_module
        return y1, y2

    @staticmethod
    def backward(ctx, dy1, dy2):
        y1, y2 = ctx.saved_tensors
        F_module, G_module = ctx.F, ctx.G

        # reconstruct x1, x2 from y1, y2
        with torch.enable_grad():
            y1_ = y1.detach().requires_grad_(True)
            y2_ = y2.detach().requires_grad_(True)

            # x2 = y2 - G(y1)
            G_out = G_module(y1_)
            x2 = y2_ - G_out

            # x1 = y1 - F(x2)
            F_out = F_module(x2)
            x1 = y1_ - F_out

            # now compute gradients through F and G
            grad_x1 = dy1.clone()
            grad_x2 = dy2.clone()

            # backprop through F: x1 = y1 - F(x2)
            grad_F_out, = torch.autograd.grad(
                outputs=F_out, inputs=x2,
                grad_outputs=-grad_x1,
                retain_graph=False, allow_unused=False,
                create_graph=False,
            )
            grad_x2 = grad_x2 + grad_F_out

            # backprop through G: x2 = y2 - G(y1)
            grad_G_out, = torch.autograd.grad(
                outputs=G_out, inputs=y1_,
                grad_outputs=-grad_x2,
                retain_graph=False, allow_unused=False,
                create_graph=False,
            )
            grad_x1 = grad_x1 + grad_G_out

        return grad_x1, grad_x2, None, None

In [16]:
import torch, time

x = torch.randint(0, 50257, (8, 512), device=cfg.device)
y = torch.randint(0, 50257, (8, 512), device=cfg.device)

model.train()
torch.cuda.synchronize()
t0 = time.time()
for _ in range(20):
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
elapsed = time.time() - t0
print(f"20 steps in {elapsed:.2f}s = {20*8*512/elapsed:.0f} tok/s (no dataloader)")

20 steps in 4.60s = 17810 tok/s (no dataloader)


In [14]:
import gc, math, torch
import torch.nn as nn
import torch.nn.functional as F

# ---- attention with fused kernel ----
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        # fused, memory-efficient, causal
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.fc2 = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.ff = FeedForward(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init)
        for pn, p in self.named_parameters():
            if pn.endswith("proj.weight") or pn.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    def num_params(self):
        return sum(p.numel() for p in self.parameters())

# ---- rebuild model + optimizer ----
if "model" in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

cfg.batch_size = 8
model = GPT(cfg).to(cfg.device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    betas=(0.9, 0.95),
    weight_decay=cfg.weight_decay,
)

# rebuild loader to match batch size
from torch.utils.data import DataLoader
loader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f"params: {model.num_params():,} ({model.num_params()/1e6:.2f}M)")
print(f"batch_size: {cfg.batch_size}")
print(f"free GPU mem: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print("ready")

params: 20,609,568 (20.61M)
batch_size: 8
free GPU mem: 14.86 GB
ready


In [17]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=4,          # was 2
    pin_memory=False,       # pinning costs CPU; skip it on T4
    drop_last=True,
    persistent_workers=True, # keep workers alive between epochs
    prefetch_factor=4,       # each worker preloads 4 batches
)
print("loader rebuilt: num_workers=4, prefetch=4, pin=False")

loader rebuilt: num_workers=4, prefetch=4, pin=False


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [18]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=False,
    drop_last=True,
    persistent_workers=True,
    prefetch_factor=4,
)
print("loader rebuilt: num_workers=2, prefetch=4, pin=False, persistent=True")

loader rebuilt: num_workers=2, prefetch=4, pin=False, persistent=True


In [20]:
import torch, numpy as np, time

tokens_i64 = np.load("/content/tokens_50m.npy").astype(np.int64)
B, T = cfg.batch_size, cfg.block_size

def get_batch():
    s = np.random.randint(0, len(tokens_i64) - T - 1)
    chunk = tokens_i64[s : s + T + 1]
    x = torch.from_numpy(chunk[:-1]).long().unsqueeze(0).repeat(B, 1)
    y = torch.from_numpy(chunk[1:]).long().unsqueeze(0).repeat(B, 1)
    return x.to(cfg.device), y.to(cfg.device)

model.train()
torch.cuda.synchronize()
t0 = time.time()
for _ in range(20):
    x, y = get_batch()
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
elapsed = time.time() - t0
print(f"{20*B*T/elapsed:.0f} tok/s with custom batch fn (no DataLoader)")

17794 tok/s with custom batch fn (no DataLoader)


In [21]:
import time, numpy as np, torch

tokens_i64 = np.load("/content/tokens_50m.npy").astype(np.int64)
B, T = cfg.batch_size, cfg.block_size

def get_batch():
    s = np.random.randint(0, len(tokens_i64) - T - 1)
    chunk = tokens_i64[s : s + T + 1]
    x = torch.from_numpy(chunk[:-1]).long().unsqueeze(0).repeat(B, 1)
    y = torch.from_numpy(chunk[1:]).long().unsqueeze(0).repeat(B, 1)
    return x.to(cfg.device), y.to(cfg.device)

def train(model, max_steps, tag="run"):
    model.train()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    t0 = time.time()
    tokens_seen = 0
    losses = []
    step = 0

    while step < max_steps:
        x, y = get_batch()

        lr = get_lr(step)
        for g in optimizer.param_groups:
            g["lr"] = lr

        logits, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        losses.append(loss.item())
        tokens_seen += x.numel()
        step += 1

        if step % cfg.log_every == 0 or step == 1:
            torch.cuda.synchronize()
            elapsed = time.time() - t0
            tps = tokens_seen / elapsed
            mem = torch.cuda.max_memory_allocated() / 1e9
            print(f"[{tag}] step {step:5d}/{max_steps}  loss {loss.item():.4f}  "
                  f"lr {lr:.2e}  {tps:.0f} tok/s  peak mem {mem:.2f} GB")

    torch.cuda.synchronize()
    total_time = time.time() - t0
    final_tps = tokens_seen / total_time
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    final_loss = sum(losses[-20:]) / len(losses[-20:])

    print(f"\n=== {tag} done ===")
    print(f"final loss (avg last 20): {final_loss:.4f}")
    print(f"total time: {total_time:.1f}s")
    print(f"tokens/sec: {final_tps:.0f}")
    print(f"peak memory: {peak_mem:.2f} GB")
    print(f"total tokens seen: {tokens_seen:,}")

    return {
        "tag": tag, "final_loss": final_loss, "total_time": total_time,
        "tokens_per_sec": final_tps, "peak_mem_gb": peak_mem,
        "tokens_seen": tokens_seen, "losses": losses,
    }

print("train() rebuilt with custom batch fn")

train() rebuilt with custom batch fn


In [23]:
import gc, torch

if "model" in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

model = GPT(cfg).to(cfg.device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    betas=(0.9, 0.95),
    weight_decay=cfg.weight_decay,
)
print(f"fresh model: {model.num_params():,}")
print(f"free GPU mem: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

fresh model: 20,609,568
free GPU mem: 12.45 GB


In [25]:
import gc, torch

cfg.batch_size = 32
cfg.max_steps = 3051   # 3051 × 32 × 512 = 50M

if "model" in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

model = GPT(cfg).to(cfg.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate,
                              betas=(0.9, 0.95), weight_decay=cfg.weight_decay)

# rebuild get_batch to use the new batch size
B, T = cfg.batch_size, cfg.block_size

print(f"params: {model.num_params():,}")
print(f"batch: {cfg.batch_size}, tokens/step: {cfg.batch_size * cfg.block_size}")
print(f"total tokens at max_steps: {cfg.max_steps * cfg.batch_size * cfg.block_size:,}")

params: 20,609,568
batch: 32, tokens/step: 16384
total tokens at max_steps: 49,987,584


In [27]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB / {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")

free: 4.89 GB / 15.64 GB
